In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [7]:
model = load_model('model.h5')

with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)
with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)


In [4]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',  
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000

}

In [8]:
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\ASUS\Desktop\dlproject1\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [10]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [19]:
print("Gender values:")
print(input_df['Gender'].unique())

print("\nEncoder classes:")
print(label_encoder_gender.classes_)

print("\nGender data type:")
print(input_df['Gender'].dtype)

Gender values:
[1]

Encoder classes:
['Female' 'Male']

Gender data type:
int64


In [20]:
print(input_df)
print(input_df.dtypes)

   CreditScore Geography  Gender  Age  Tenure  Balance  NumOfProducts  \
0          600    France       1   40       3    60000              2   

   HasCrCard  IsActiveMember  EstimatedSalary  
0          1               1            50000  
CreditScore        int64
Geography            str
Gender             int64
Age                int64
Tenure             int64
Balance            int64
NumOfProducts      int64
HasCrCard          int64
IsActiveMember     int64
EstimatedSalary    int64
dtype: object


In [32]:
#concatination one hot encoded
input_df=pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)


KeyError: "['Geography'] not found in axis"

In [28]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [34]:
print("Expected columns:")
print(scaler.feature_names_in_.tolist())

print("\nCurrent input columns:")
print(input_df.columns.tolist())

Expected columns:
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Gender_0', 'Gender_1']

Current input columns:
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain']


In [35]:
input_df['Gender_0'] = 0
input_df['Gender_1'] = 1

In [36]:
input_df = input_df.drop(columns=['Gender'], errors='ignore')

In [37]:
input_df = input_df.reindex(
    columns=scaler.feature_names_in_,
    fill_value=0
)

input_scaled = scaler.transform(input_df)

In [38]:
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step


array([[0.03726302]], dtype=float32)

In [39]:
prediction_probability = prediction[0][0]
prediction_probability


np.float32(0.037263017)

In [40]:
if prediction_probability > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is unlikely to churn.")

The customer is unlikely to churn.
